In [1]:
"""
🟢 A-2. 놀이터 — 내가 만든 문장으로 모델을 찔러 본다  🎮 (가벼움 · 목표 30분)

앞의 미션들이 "만들고 고치는" 것이었다면, 이건 **가지고 노는** 미션이다.
훈련된 모델에게 내가 쓴 리뷰를 직접 먹여 보고, 언제 맞히고 언제 틀리는지 사냥한다.

세 가지를 한다.
  (a) 내가 쓴 영어 리뷰를 예측시켜 본다
  (b) 모델이 '틀린' 리뷰를 찾아 직접 읽어 본다  ← 왜 틀렸을까?
  (c) ★ 반전을 문장 앞에 둘 때 vs 뒤에 둘 때  ← 오늘 배운 것의 결정적 확인

⚠️ 정확도가 0.68인 모델이다. **틀리는 게 정상이다.** 틀린 걸 찾아내는 게 이 미션의 재미다.
   "왜 틀렸을까"를 오늘 배운 것(마지막 은닉만 본다·앞을 잊는다)으로 설명해 보자.
"""

'\n🟢 A-2. 놀이터 — 내가 만든 문장으로 모델을 찔러 본다  🎮 (가벼움 · 목표 30분)\n\n앞의 미션들이 "만들고 고치는" 것이었다면, 이건 **가지고 노는** 미션이다.\n훈련된 모델에게 내가 쓴 리뷰를 직접 먹여 보고, 언제 맞히고 언제 틀리는지 사냥한다.\n\n세 가지를 한다.\n  (a) 내가 쓴 영어 리뷰를 예측시켜 본다\n  (b) 모델이 \'틀린\' 리뷰를 찾아 직접 읽어 본다  ← 왜 틀렸을까?\n  (c) ★ 반전을 문장 앞에 둘 때 vs 뒤에 둘 때  ← 오늘 배운 것의 결정적 확인\n\n⚠️ 정확도가 0.68인 모델이다. **틀리는 게 정상이다.** 틀린 걸 찾아내는 게 이 미션의 재미다.\n   "왜 틀렸을까"를 오늘 배운 것(마지막 은닉만 본다·앞을 잊는다)으로 설명해 보자.\n'

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import sys
from pathlib import Path
try:
    _HERE = Path(__file__).resolve().parent
except NameError:                      # 노트북 셀에는 __file__ 이 없다
    _HERE = Path.cwd()
sys.path.insert(0, str(_HERE.parent))  # day03/ 의 모듈을 쓰기 위해

from imdb_data import load_imdb
from textutils import tokenize_en, build_vocab, pad_and_tensor
from model import IMDBRnn

torch.manual_seed(42)
VOCAB_SIZE, MAX_LEN, EMBED, HIDDEN = 2000, 100, 32, 32

## 준비 — 모델을 하나 훈련시킨다 (1분 안팎)

05_train.py 와 같은 내용이다. 놀려면 먼저 모델이 있어야 하니 여기서 한 번 훈련한다.

In [3]:
train, val = load_imdb(n_train=5000, n_val=2000)
train_toks = [tokenize_en(t) for t in train["text"]]
val_toks = [tokenize_en(t) for t in val["text"]]
word2idx, _ = build_vocab(train_toks, max_size=VOCAB_SIZE)

X_train = pad_and_tensor(train_toks, word2idx, MAX_LEN)
y_train = torch.tensor(train["label"], dtype=torch.float32)
X_val = pad_and_tensor(val_toks, word2idx, MAX_LEN)
y_val = torch.tensor(val["label"], dtype=torch.float32)

loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
model = IMDBRnn(len(word2idx), EMBED, HIDDEN)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.BCELoss()

best_acc, best_state = 0.0, None
for epoch in range(9):                     # 최고점 부근(9)까지만 돌린다
    model.train()
    for xb, yb in loader:
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = ((model(X_val) > 0.5).float() == y_val).float().mean().item()
    if acc > best_acc:
        best_acc, best_state = acc, {k: v.clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
model.eval()
print(f"준비 완료 — 검증 정확도 {best_acc:.3f}")

준비 완료 — 검증 정확도 0.655


In [4]:
def predict(text):
    """영어 문장 하나를 넣으면 '긍정일 확률'을 돌려준다.
    §4의 파이프라인(토큰화 → 인코딩 → 앞쪽 패딩)을 그대로 통과시킨다."""
    toks = tokenize_en(text)                              # 소문자 + 알파벳 단어만
    x = pad_and_tensor([toks], word2idx, MAX_LEN)         # (1, 100) — 앞쪽 패딩
    with torch.no_grad():
        return model(x).item()                            # 0~1 확률


def show(text, label=""):
    p = predict(text)
    verdict = "긍정" if p > 0.5 else "부정"
    print(f"  [{verdict} {p:.2f}] {label}{text[:70]}{'…' if len(text) > 70 else ''}")
    return p

## (a) 내가 쓴 리뷰로 예측시켜 보기

아래 문장을 **마음대로 바꿔 가며** 실행해 보자. 어떤 단어를 넣으면 확률이 확 움직이는가?

해 볼 것:
- 확실한 칭찬 / 확실한 혹평
- `not` 을 넣어 뒤집어 보기 (`good` → `not good`) — 모델이 부정어를 이해할까?
- 아주 짧은 문장 vs 아주 긴 문장

In [5]:
print("(a) 내가 쓴 리뷰")
show("this movie was absolutely wonderful and i loved every minute of it")
show("terrible film boring acting and a complete waste of time")
show("this movie was good")
show("this movie was not good")          # ← not 을 이해할까?
show("i expected a masterpiece but got a mess")

(a) 내가 쓴 리뷰
  [긍정 0.70] this movie was absolutely wonderful and i loved every minute of it
  [부정 0.11] terrible film boring acting and a complete waste of time
  [부정 0.42] this movie was good
  [긍정 0.50] this movie was not good
  [긍정 0.56] i expected a masterpiece but got a mess


0.5555809140205383

## (b) 모델이 틀린 리뷰 사냥하기

정확도 0.68 = **10편 중 3편은 틀린다.** 그 3편을 직접 찾아 읽어 보자.
틀린 리뷰를 읽으면 모델이 무엇을 못 하는지가 보인다.

In [6]:
print("\n(b) 모델이 틀린 리뷰 — 확신에 차서 틀린 것부터")
with torch.no_grad():
    probs = model(X_val)
pred_label = (probs > 0.5).float()
wrong = (pred_label != y_val).nonzero(as_tuple=True)[0]

# '틀렸는데 확신까지 했던' 순서로 정렬 = 가장 흥미로운 실패
confidence = (probs[wrong] - 0.5).abs()
worst = wrong[confidence.argsort(descending=True)][:3]

print(f"틀린 개수: {len(wrong)} / {len(y_val)}")
for i in worst:
    i = i.item()
    truth = "긍정" if y_val[i] == 1 else "부정"
    print(f"\n  정답={truth} 인데 모델은 {probs[i]:.2f} 로 반대라고 확신")
    print(f"  원문: {val['text'][i][:200].strip()}…")


(b) 모델이 틀린 리뷰 — 확신에 차서 틀린 것부터
틀린 개수: 689 / 2000

  정답=긍정 인데 모델은 0.07 로 반대라고 확신
  원문: Project A II is a classic Jackie Chan movie with all the kung fu, crazy stunts and slapstick humor you expect. Not as good as the prequel but still it is a great movie if you just want something fun t…

  정답=긍정 인데 모델은 0.08 로 반대라고 확신
  원문: Chris Penn is hilarious as the all-time stoner brother of Jeff spicoli. This movie is great because it was a lot more real and funnier than fast times at ridgemont high. Casting was perfect and one of…

  정답=긍정 인데 모델은 0.08 로 반대라고 확신
  원문: Since watching the trailer in "The Little Mermaid II: Return To The Sea" DVD, I had a feeling that this movie is gonna be great 'cause I am a huge Disney fan. And guess what? I'm right! This movie is…


> **생각해 볼 것**: 위 리뷰들을 읽어 보면 대개 이렇다 —
> 앞에서 길게 칭찬하다 마지막에 뒤집거나, 비꼬거나, 줄거리 설명이 길다.
> 우리 모델은 **마지막 은닉 상태 하나**로 판단한다는 걸 떠올리자.

## (c) ★ 반전을 앞에 둘까, 뒤에 둘까 — 오늘의 결정적 실험

같은 내용을 순서만 바꿔 넣는다. 모델이 **어느 쪽 반전을 더 잘 잡는지** 보자.
오늘 배운 것이 맞다면 — RNN은 마지막 은닉으로 판단하므로 **뒤쪽 반전에 더 민감**해야 한다.

In [7]:
print("\n(c) 반전의 위치를 바꾸면")
GOOD = "the acting was great and the story was beautiful and i enjoyed it so much"
BAD = "but honestly it was terrible boring and a complete waste of time"

p_end = show(f"{GOOD} {BAD}", label="반전이 뒤 → ")
p_front = show(f"{BAD} {GOOD}", label="반전이 앞 → ")

print(f"\n  뒤에 부정을 두면: {p_end:.2f}   앞에 부정을 두면: {p_front:.2f}")
print("  → 마지막에 온 쪽 감정으로 기울면, 그게 '마지막 은닉만 본다'는 증거다.")


(c) 반전의 위치를 바꾸면
  [부정 0.11] 반전이 뒤 → the acting was great and the story was beautiful and i enjoyed it so m…
  [부정 0.40] 반전이 앞 → but honestly it was terrible boring and a complete waste of time the a…

  뒤에 부정을 두면: 0.11   앞에 부정을 두면: 0.40
  → 마지막에 온 쪽 감정으로 기울면, 그게 '마지막 은닉만 본다'는 증거다.


## 기록하고 이야기하자

| 실험 | 내가 넣은 문장 | 확률 | 맞았나? | 왜 그럴까 |
|---|---|---|---|---|
| (a) | | | | |
| (b) | | | | |
| (c) | | | | |

**회고 때 나눌 것**
1. 가장 어이없게 틀린 문장은? (제일 웃긴 실패를 공유하자)
2. `not` 을 넣었을 때 모델이 뒤집혔는가? 안 뒤집혔다면 왜일까?
3. (c)에서 앞/뒤 어느 쪽이 더 셌는가? 오늘 배운 것과 맞는가?
4. 이 한계를 고치려면 무엇이 필요할까? → **내일(Day 4) LSTM**